# Part II walkthrough — when applying is not enough

Part I did everything with two primitives: apply an operator, then round. That is
exactly enough for an explicit scheme and nothing else. This notebook shows why that
runs out, and builds the two capabilities that replace it:

1. **solving** $Ax = b$ on the tensor-train manifold, and
2. **constructing** a train from a function you can only sample.

The second is what stops the whole programme being circular: without it you would need
the $2^n$-vector in order to compress the $2^n$-vector.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_quimb as qq
import qtade_tn as tn

plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 10})

## 1. Quantics gives you resolution; explicit stepping takes it away

The explicit stability bound is $\alpha\Delta t/\Delta x^2 \le 1/2$ with
$\Delta x = 2^{-n}$, so $\Delta t \sim 4^{-n}$. Memory grows like $n$; the number of
steps grows like $4^n$. Look at what that means.

In [ ]:
print(f"{'n':>3} {'N':>12} {'memory ~ n*chi^2':>18} {'steps to t = 0.1':>20}")
for n in (10, 15, 20, 25, 30):
    dt = 0.5 * (2.0 ** -n) ** 2
    print(f"{n:>3} {2**n:>12,d} {n * 100:>18,d} {int(0.1 / dt):>20,d}")

At $n=20$ you can *store* the field on a laptop and would still need $10^{11}$ steps.
Compression bought resolution and the explicit scheme charges for it in time. The only
way out is an implicit scheme, and an implicit scheme needs a solver.

## 2. Solving $Ax = b$ without ever inverting $A$

$A$ is an MPO, $b$ an MPS, and we want $x$ as an MPS. Do **not** form $A^{-1}$: it has
no reason to be low rank even when $A$ and $x$ both are. Instead minimise over the
manifold, one core at a time.

Backward Euler for the heat equation: $(\mathbb{1} - \alpha\Delta t\,L)\,u^{k+1} = u^k$.

In [ ]:
n, alpha = 14, 1.0
h = 2.0 ** -n
dt = 1e-5                            # 10^5 times the explicit limit
xs = np.linspace(0, 1, 2 ** n, endpoint=False)

L = tn.qtt_laplacian(n, dx=h)
A = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, -alpha * dt)), 1e-13)
u0 = tn.qtt_from_vector(np.exp(-((xs - 0.5) / 0.05) ** 2), eps=1e-10)

print(f"explicit limit dt = {0.5 * h**2:.2e};  we are using dt = {dt:.0e}")
print(f"A: MPO rank {max(tn.mpo_ranks(A))}, built analytically — no assembly step")
print(f"b: MPS rank {max(tn.tt_ranks(u0))}\n")

print("two-site DMRG, cold start:")
x_dmrg = tn.dmrg_solve(A, u0, sweeps=4, eps=1e-10, chi_max=64, verbose=True)

### One-site ALS: the same idea with one thing removed

ALS freezes all cores but one and solves exactly. It cannot change $\chi$. Watch what
that costs when the fixed rank is too small.

In [ ]:
for chi in (4, 8, 16, 32):
    guess = tn.tt_round([c.copy() for c in u0], eps=1e-14, chi_max=chi)
    xa = tn.als_solve(A, u0, x0=guess, sweeps=6)
    print(f"ALS at fixed chi = {chi:>2}: residual {tn.residual(A, xa, u0):.2e}")
print(f"\ntwo-site DMRG chose chi = {max(tn.tt_ranks(x_dmrg))} by itself, "
      f"residual {tn.residual(A, x_dmrg, u0):.2e}")

The one-site residual plateaus at a level set by the rank you guessed, and no number of
sweeps improves it. That single limitation is the whole reason two-site DMRG and AMEn
exist: the truncated SVD inside a two-site update *chooses* the rank from a tolerance.

## 3. Warm starting is the largest practical lever

Consecutive timesteps differ by $O(\Delta t)$, so the previous solution is an excellent
initial guess. This is the main reason the observed cost exponent is near $\chi^{4}$
rather than the worst-case $\chi^{6}$.

In [ ]:
u = u0
cold_t, warm_t = [], []
for k in range(6):
    t0 = time.perf_counter()
    _ = tn.dmrg_solve(A, u, sweeps=2, eps=1e-10, chi_max=64)
    cold_t.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    u = tn.dmrg_solve(A, u, x0=u, sweeps=1, eps=1e-10, chi_max=64)
    warm_t.append(time.perf_counter() - t0)

print(f"cold start, 2 sweeps: {np.mean(cold_t):.3f} s/step")
print(f"warm start, 1 sweep : {np.mean(warm_t):.3f} s/step  "
      f"({np.mean(cold_t) / np.mean(warm_t):.1f}x faster)")
print("\nThe warm start is not a micro-optimisation: consecutive solutions differ by")
print("O(dt), so one sweep from the previous answer beats several from scratch. This is")
print("the single largest reason the measured cost exponent is near chi^4 and not chi^6.")

## 4. The error budget now has three terms, not two

1. time discretisation, $O(\Delta t)$
2. space discretisation, $O(\Delta x^2)$
3. **the solver residual** — new, and entirely under your control

Keep (3) well below (1) and (2). Otherwise you converge efficiently to the wrong answer,
which is the least useful of all possible outcomes.

In [ ]:
# Heat equation with a known exact solution: one Fourier mode decays as exp(-alpha k^2 t).
# To see the solver term in the budget we have to *degrade the solver on purpose*, which
# we do by capping the bond dimension it is allowed to use.
n = 12
h = 2.0 ** -n
xs = np.linspace(0, 1, 2 ** n, endpoint=False)
kmode = 2 * np.pi
L = tn.qtt_laplacian(n, dx=h)
exact = lambda t: np.sin(kmode * xs) * np.exp(-kmode ** 2 * t)

T = 0.005
print(f"{'dt':>10} {'solver chi_max':>15} {'solver resid':>14} {'error vs exact':>16}")
for dt in (T / 4, T / 64):
    A = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, -dt)), 1e-13)
    for chi_cap in (2, 4, 40):
        u = tn.qtt_from_vector(exact(0.0), eps=1e-12)
        last = 0.0
        for _ in range(int(round(T / dt))):
            prev = u
            u = tn.dmrg_solve(A, prev, x0=prev, sweeps=1, eps=1e-12, chi_max=chi_cap)
            last = tn.residual(A, u, prev)
        err = np.linalg.norm(tn.qtt_to_vector(u) - exact(T)) / np.linalg.norm(exact(T))
        print(f"{dt:>10.2e} {chi_cap:>15} {last:>14.2e} {err:>16.2e}")

Read the table in blocks of three. At the coarse $\Delta t$, tightening the solver from
$\chi=4$ to $\chi=40$ changes the answer not at all: the time discretisation dominates,
and every extra sweep is wasted. At the fine $\Delta t$ the sloppy solve is suddenly
what limits you.

The practical rule: target a residual about an order of magnitude below the expected
discretisation error, and no tighter. Going further is pure cost; not going far enough
means you converge efficiently to the wrong answer, which is the least useful of all
possible outcomes.

## 5. TT-cross: building a train from samples

Three routes to a quantics train, and only one of them scales:

| route | cost | needs |
|---|---|---|
| analytic cores | free | a closed form |
| TT-SVD | $O(2^n)$ | the dense vector |
| TT-cross | $O(n\chi^2)$ evaluations | a black box $f$ |

`tt_cross` takes a function of an integer index array. It never sees a grid.

In [ ]:
n = 20
weights = 2.0 ** -(np.arange(n) + 1)


def f_black_box(idx):
    """f evaluated at binary multi-indices. Could be a lookup, a solver, anything."""
    xv = idx @ weights
    return np.exp(-2 * xv) + np.sin(6 * np.pi * xv)


cores, stats = tn.tt_cross(f_black_box, [2] * n, rank=6, sweeps=3, return_stats=True)
xs = np.linspace(0, 1, 2 ** n, endpoint=False)
exact_v = np.exp(-2 * xs) + np.sin(6 * np.pi * xs)
err = np.linalg.norm(tn.qtt_to_vector(cores) - exact_v) / np.linalg.norm(exact_v)
print(f"grid points:      {2**n:,d}")
print(f"f evaluations:    {stats['evaluations']:,d}  "
      f"({100 * stats['evaluations'] / 2**n:.3f}% of the grid)")
print(f"bond dimensions:  {tn.tt_ranks(cores)}")
print(f"relative error:   {err:.2e}")

Machine precision from a fifth of a percent of the grid. The mechanism is **maxvol**:
a rank-$r$ matrix is exactly reconstructible from $r$ well-chosen rows and columns, and
maxvol chooses them greedily by maximising $|\det|$.

In [ ]:
rng = np.random.default_rng(0)
Q, _ = np.linalg.qr(rng.standard_normal((400, 8)))
rows = tn.maxvol(Q)
B = Q @ np.linalg.inv(Q[rows])
print(f"maxvol picked rows {np.sort(rows)}")
print(f"max |entry| of Q inv(Q[rows]) = {np.abs(B).max():.3f}   "
      f"(bounded near 1: that bound is why cross interpolation is stable)")

### The failure modes, on purpose

Cross interpolation is *exact* on the fibres it samples. That is a virtue for clean
data and a liability for dirty data.

In [ ]:
n = 16
w = 2.0 ** -(np.arange(n) + 1)
xs = np.linspace(0, 1, 2 ** n, endpoint=False)
noise_field = 0.01 * np.random.default_rng(2).standard_normal(2 ** n)

cases = {
    "clean smooth f": (lambda idx: np.exp(-2 * (idx @ w)), np.exp(-2 * xs)),
    "f + 1% noise": (
        lambda idx: np.exp(-2 * (idx @ w)) + noise_field[
            (idx * (2 ** (n - 1 - np.arange(n)))).sum(axis=1)],
        np.exp(-2 * xs) + noise_field),
    "spike, width 1e-3": (
        lambda idx: np.exp(-((idx @ w - 0.371) / 1e-3) ** 2),
        np.exp(-((xs - 0.371) / 1e-3) ** 2)),
    "spike, width 1e-5": (
        lambda idx: np.exp(-((idx @ w - 0.6180339887) / 1e-5) ** 2),
        np.exp(-((xs - 0.6180339887) / 1e-5) ** 2)),
}
for name, (fn, ref) in cases.items():
    for rank in (8, 16):
        c = tn.tt_cross(fn, [2] * n, rank=rank, sweeps=3)
        got = tn.qtt_to_vector(c)
        print(f"{name:>20} rank {rank:>2}: rel err = "
              f"{np.linalg.norm(got - ref) / np.linalg.norm(ref):.2e}   "
              f"max |value found| = {np.abs(got).max():.3f} (true {np.abs(ref).max():.1f})")

Three separate lessons in that table.

* The noisy case is reproduced **faithfully, noise included**. Cross interpolation is
  exact on the fibres it samples; it does not denoise, it interpolates. That is a
  virtue for clean data and a liability for dirty data.
* The width-$10^{-3}$ spike is found without difficulty. Cross is more robust than the
  folklore suggests, and a moderately localised feature is not a problem.
* The width-$10^{-5}$ spike — about one grid cell wide, at an irrational position — is
  **missed completely**, at every rank. The returned function is identically zero and
  the relative error is 100%.

The last row is the one to remember, and note *how* it fails. No pivot ever lands near
the feature, so the method never learns it exists, the cross error it reports is small,
and nothing in the output says anything is wrong. Rank-adaptive variants (TCI, `xfac`)
improve the odds by adding pivots where the local error is largest, but there is no
guarantee for an adversarial black box. If you know where the interesting region is,
tell the algorithm.

## 6. Geometry: buying off a curved boundary

Part I measured the problem: a disc indicator has a bond dimension that grows with
resolution, because a circle crosses every length scale at once. The fix in Peddinti
*et al.* (2024) is to give up the sharp edge deliberately, with a knob and an error bar:

$$m(x) = 1 - e^{-\alpha (q(x) + |q(x)|)}, \qquad q|_{\partial\Omega} = 0.$$

$q + |q|$ vanishes identically where $q<0$, so $m$ is *exactly* zero inside the body —
no approximation is made in the interior — and rises smoothly outside it.

In [ ]:
m = 9
N = 2 ** m
xx = np.linspace(0, 1, N, endpoint=False)
X, Y = np.meshgrid(xx, xx, indexing="ij")
q = 0.25 ** 2 - ((X - 0.5) ** 2 + (Y - 0.5) ** 2)
indicator = (q > 0).astype(float)

print(f"{'alpha':>8} {'chi @ 1e-1':>11} {'chi @ 1e-2':>11} {'chi @ 1e-6':>11} "
      f"{'L2 to indicator':>17}")
print(f"{'sharp':>8} " + " ".join(
    f"{max(tn.tt_ranks(qq.from_grid(indicator, eps=e))):>11}"
    for e in (1e-1, 1e-2, 1e-6)) + f" {0.0:>17.4f}")
for a in (50, 200, 1000, 5000):
    mask = 1 - np.exp(-a * (q + np.abs(q)))
    chis = [max(tn.tt_ranks(qq.from_grid(mask, eps=e))) for e in (1e-1, 1e-2, 1e-6)]
    dist = np.linalg.norm(mask - indicator) / np.linalg.norm(indicator)
    print(f"{a:>8} " + " ".join(f"{c:>11}" for c in chis) + f" {dist:>17.4f}")

Two things to take from that table, and the second is the one people get wrong.

* The distance to the true indicator falls like $\alpha^{-1/2}$ — check the column:
  each fourfold increase in $\alpha$ roughly halves it. That is the bound
  $\sqrt{\ell_0^{D-2}/\alpha}$ from the paper, and it is what makes this a *modelling
  choice with an error bar* rather than a hack.
* Smoothing does **not** reduce the rank at tight tolerance. Compare the `chi @ 1e-6`
  column: the smooth mask is a different function with its own structure and it is no
  cheaper to represent exactly. The saving appears at the accuracy you actually want —
  look at `chi @ 1e-1` and `chi @ 1e-2`.

So the honest statement is not "sharp edges have high rank so smooth them". It is:
*for a given accuracy budget, the smoothed mask buys you a lower rank, and $\alpha$
lets you price the trade.*

In [ ]:
mask = 1 - np.exp(-1000 * (q + np.abs(q)))
fig, ax = plt.subplots(1, 3, figsize=(10, 3))
ax[0].imshow(indicator.T, origin="lower", cmap="gray_r")
ax[0].set_title("sharp indicator")
ax[1].imshow(mask.T, origin="lower", cmap="gray_r")
ax[1].set_title(r"smoothed, $\alpha = 1000$")
ax[2].plot(xx, indicator[:, N // 2], label="sharp")
ax[2].plot(xx, mask[:, N // 2], label=r"$\alpha=1000$")
ax[2].set(xlim=(0.2, 0.35), title="the edge, close up")
ax[2].legend()
for a_ in ax[:2]:
    a_.set_xticks([]), a_.set_yticks([])
plt.tight_layout()

The mask is applied by an elementwise (Hadamard) product with the velocity field at
every step: no-slip without ever meshing the body, and no immersed-boundary force term.
That is the last piece we need before Part III.